In [4]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql import functions as F

## Question 1: Install Spark and PySpark
Run PySpark, create a local spark session, execute spark.version

In [5]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [6]:
# Question 1 Answer
spark.version

'4.1.1'

## Question 2: Yellow November 2025 - File Size
Read the November 2025 Yellow taxi data, repartition to 4 partitions, and save as parquet.
What is the average size of the parquet files?

In [7]:
# Check if file exists
!ls -lh yellow_tripdata_2025-11.parquet

-rw-r--r-- 1 muham 197609 68M Mar 12 23:12 yellow_tripdata_2025-11.parquet


In [8]:
# Read the parquet file
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

# Repartition to 4 partitions
df = df.repartition(4)

# Save to parquet
df.write.mode('overwrite').parquet('data/pq/yellow/2025/11/')

In [9]:
# Check the parquet files created
!ls -lh data/pq/yellow/2025/11/*.parquet

-rw-r--r-- 1 muham 197609 25M Mar 12 23:36 data/pq/yellow/2025/11/part-00000-b31a4ccb-03e3-4134-9a1c-f07d02e33be2-c000.snappy.parquet
-rw-r--r-- 1 muham 197609 25M Mar 12 23:36 data/pq/yellow/2025/11/part-00001-b31a4ccb-03e3-4134-9a1c-f07d02e33be2-c000.snappy.parquet
-rw-r--r-- 1 muham 197609 25M Mar 12 23:36 data/pq/yellow/2025/11/part-00002-b31a4ccb-03e3-4134-9a1c-f07d02e33be2-c000.snappy.parquet
-rw-r--r-- 1 muham 197609 25M Mar 12 23:36 data/pq/yellow/2025/11/part-00003-b31a4ccb-03e3-4134-9a1c-f07d02e33be2-c000.snappy.parquet


In [10]:
# Calculate average file size
import os

parquet_files = []
for root, dirs, files in os.walk('data/pq/yellow/2025/11/'):
    for file in files:
        if file.endswith('.parquet'):
            filepath = os.path.join(root, file)
            size_mb = os.path.getsize(filepath) / (1024 * 1024)
            parquet_files.append(size_mb)
            print(f"{file}: {size_mb:.2f} MB")

avg_size = sum(parquet_files) / len(parquet_files)
print(f"\nAverage parquet file size: {avg_size:.2f} MB")
print(f"Question 2 Answer: ~{int(round(avg_size))} MB")

part-00000-b31a4ccb-03e3-4134-9a1c-f07d02e33be2-c000.snappy.parquet: 24.41 MB
part-00001-b31a4ccb-03e3-4134-9a1c-f07d02e33be2-c000.snappy.parquet: 24.40 MB
part-00002-b31a4ccb-03e3-4134-9a1c-f07d02e33be2-c000.snappy.parquet: 24.43 MB
part-00003-b31a4ccb-03e3-4134-9a1c-f07d02e33be2-c000.snappy.parquet: 24.43 MB

Average parquet file size: 24.42 MB
Question 2 Answer: ~24 MB


In [11]:
# Re-read the parquet data for subsequent questions
df = spark.read.parquet('data/pq/yellow/2025/11/')

## Question 3: How many taxi trips were there on November 15, 2025?

In [15]:
df \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .filter("pickup_date = '2025-11-15'") \
    .count()

162604

In [16]:
# Register as temp table for SQL queries
df.createOrReplaceTempView('trips_data')

In [17]:
spark.sql("""
SELECT
    COUNT(1)
FROM 
    trips_data
WHERE
    DATE(tpep_pickup_datetime) = '2025-11-15'
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



## Question 4: Longest trip in hours

In [18]:
# Check available columns
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [21]:
spark.sql("""
SELECT
    (UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 3600 AS trip_duration_hours
FROM 
    trips_data
ORDER BY
    trip_duration_hours DESC
LIMIT 5
""").show()

+-------------------+
|trip_duration_hours|
+-------------------+
|  90.64666666666666|
|  76.94833333333334|
|  76.21388888888889|
|  69.28861111111111|
|  67.08055555555555|
+-------------------+



## Question 5: Spark User Interface Port

Spark's User Interface which shows the application's dashboard runs on which local port?

**Answer: 4040**

While this notebook is running, open your browser and go to:
- http://localhost:4040

In [22]:
print("Question 5 Answer: 4040")
print("\nSpark UI: http://localhost:4040")

Question 5 Answer: 4040

Spark UI: http://localhost:4040


## Question 6: Least frequent pickup location zone

In [23]:
# Load taxi zone lookup CSV
df_zones = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv('taxi_zone_lookup.csv')

In [24]:
# Check zone columns
df_zones.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

In [25]:
# Show sample zones
df_zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [26]:
# Register zones as temp table
df_zones.createOrReplaceTempView('lookup')

In [27]:
# Find LEAST frequent pickup zones using SQL
spark.sql("""
SELECT
    lookup.Zone,
    COUNT(1) as num_trips
FROM 
    trips_data 
    INNER JOIN lookup ON lookup.LocationID = trips_data.PULocationID
GROUP BY 
    lookup.Zone
ORDER BY
    num_trips ASC
LIMIT 10
""").show(truncate=False)

+---------------------------------------------+---------+
|Zone                                         |num_trips|
+---------------------------------------------+---------+
|Arden Heights                                |1        |
|Eltingville/Annadale/Prince's Bay            |1        |
|Governor's Island/Ellis Island/Liberty Island|1        |
|Port Richmond                                |3        |
|Rikers Island                                |4        |
|Rossville/Woodrow                            |4        |
|Great Kills                                  |4        |
|Green-Wood Cemetery                          |4        |
|Jamaica Bay                                  |5        |
|Westerleigh                                  |12       |
+---------------------------------------------+---------+



In [28]:
# Alternative method using DataFrame API
pickup_counts = df.groupBy('PULocationID') \
    .count() \
    .withColumnRenamed('PULocationID', 'LocationID')

pickup_with_zones = pickup_counts.join(
    df_zones,
    'LocationID',
    'inner'
).select('Zone', 'Borough', 'count')

print("10 LEAST frequent pickup zones:")
pickup_with_zones.orderBy('count').show(10, truncate=False)

10 LEAST frequent pickup zones:
+---------------------------------------------+-------------+-----+
|Zone                                         |Borough      |count|
+---------------------------------------------+-------------+-----+
|Arden Heights                                |Staten Island|1    |
|Eltingville/Annadale/Prince's Bay            |Staten Island|1    |
|Governor's Island/Ellis Island/Liberty Island|Manhattan    |1    |
|Port Richmond                                |Staten Island|3    |
|Green-Wood Cemetery                          |Brooklyn     |4    |
|Rossville/Woodrow                            |Staten Island|4    |
|Great Kills                                  |Staten Island|4    |
|Rikers Island                                |Bronx        |4    |
|Jamaica Bay                                  |Queens       |5    |
|Westerleigh                                  |Staten Island|12   |
+---------------------------------------------+-------------+-----+
only showing top

## Summary of Answers

- **Question 1:** Check spark.version output above
- **Question 2:** Check average parquet file size (6MB, 25MB, 75MB, or 100MB)
- **Question 3:** Check trip count for November 15, 2025 (62,610 / 102,340 / 162,604 / 225,768)
- **Question 4:** Check longest trip hours (22.7 / 58.2 / 90.6 / 134.5)
- **Question 5:** 4040
- **Question 6:** Check least frequent zone (Governor's Island / Arden Heights / Rikers Island / Jamaica Bay)

In [29]:
lookup_df = spark.read.option("header", "true").csv('taxi_zone_lookup.csv')

lookup_df.registerTempTable('lookup')  

spark.sql("""

SELECT lookup.Zone , count(1) as num_trips FROM trips_data 
INNER JOIN lookup ON lookup.LocationID = trips_data.PULocationID
GROUP BY lookup.Zone
ORDER BY num_trips ASC;
""").show()  

C:\Users\muham\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\sql\classic\dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


+--------------------+---------+
|                Zone|num_trips|
+--------------------+---------+
|       Arden Heights|        1|
|Eltingville/Annad...|        1|
|Governor's Island...|        1|
|       Port Richmond|        3|
|       Rikers Island|        4|
|   Rossville/Woodrow|        4|
|         Great Kills|        4|
| Green-Wood Cemetery|        4|
|         Jamaica Bay|        5|
|         Westerleigh|       12|
|New Dorp/Midland ...|       14|
|       West Brighton|       14|
|             Oakwood|       14|
|        Crotona Park|       14|
|       Willets Point|       15|
|Breezy Point/Fort...|       16|
|Saint George/New ...|       17|
|       Broad Channel|       18|
|     Mariners Harbor|       21|
|Heartland Village...|       22|
+--------------------+---------+
only showing top 20 rows
